# FusionTrack — Multi-Object Tracking walkthrough

This notebook walks through the **Global Nearest-Neighbour (GNN)** multi-object tracker step by step, using the 3-target crossing scenario from `src/mot.py`.

Goals:
1. Understand track **birth → TENTATIVE → CONFIRMED → DELETED** lifecycle with printed numbers.
2. See the cost matrix and gating in action.
3. Inspect the **crossing at frame 49** where two targets are inside each other's chi-square gate.
4. Confirm zero ID switches and examine per-track RMSE.
5. Produce the inline static trajectory plot.

Run from the project root, or add it to `PYTHONPATH` (the first cell does that defensively).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import math
import numpy as np
from numpy.random import default_rng

from src.mot import (
    CHI2_GATE_2DOF_99,
    HITS_TO_CONFIRM,
    MAX_MISSES_BEFORE_DELETE,
    MAX_TENTATIVE_AGE,
    Track,
    TrackState,
    TrackerManager,
    run_mot_demo,
    plot_mot_results,
    compute_mot_metrics,
)
from src import mot_sim

print(f"Gate threshold chi²(2DOF, 99%): {CHI2_GATE_2DOF_99}")
print(f"Hits to confirm:                {HITS_TO_CONFIRM}")
print(f"Max misses before delete:       {MAX_MISSES_BEFORE_DELETE}")
print(f"Max tentative age (frames):     {MAX_TENTATIVE_AGE}")

## The crossing scenario — 3 straight-line CV targets

`make_crossing_scenario` generates three constant-velocity trajectories that converge near frame 49. At the crossing, Targets 1 and 2 pass within ~9 m of each other — well inside each other's chi-square gate. This is the stress test for the Hungarian assignment.

In [ ]:
trajectories = mot_sim.make_crossing_scenario(n_frames=100)
print(f"Number of targets: {len(trajectories)}")
for i, traj in enumerate(trajectories):
    print(f"  Target {i+1}: start={traj[0].round(1)}, frame-49={traj[49].round(1)}, end={traj[-1].round(1)}")

# Inter-target distances at frame 49
print("\nInter-target distances at frame 49 (the crossing):")
for i in range(len(trajectories)):
    for j in range(i+1, len(trajectories)):
        d = np.linalg.norm(trajectories[i][49] - trajectories[j][49])
        print(f"  T{i+1}–T{j+1}: {d:.1f} m")

## Track lifecycle — manual walk-through of frames 0–4

We run the `TrackerManager` frame by frame and print the track list after each step. Watch:
- Frame 0: measurements arrive, tracks are born as **TENTATIVE** with zero velocity.
- Frame 1: measurements match the tentative tracks → second hit → **CONFIRMED**.
- Any unmatched tentative track would be pruned after `MAX_TENTATIVE_AGE` frames.

In [ ]:
rng_walk = default_rng(42)
per_frame_meas = mot_sim.generate_multi_target_measurements(trajectories, rng_walk)

mgr = TrackerManager(dt=1.0)

for k in range(5):
    mgr.predict_all()
    zs = [np.array([r.range_m, r.azimuth_rad]) for r in per_frame_meas[k]]
    mgr.update(zs)
    print(f"\nFrame {k:02d}  ({len(zs)} measurements)")
    for t in mgr.tracks:
        xy = t.get_position().round(1)
        print(f"  Track {t.track_id:2d}  state={t.state.name:<10s}  hits={t.hits}  misses={t.misses}  pos={xy}")

## Cost matrix and gating at frame 2

The cost matrix is `(n_tracks × n_measurements)`. Each entry is either the **Mahalanobis distance²** in polar space (if within the chi-square gate) or `_LARGE_COST = 1e9` (gated out). The Hungarian algorithm runs on this matrix to find the globally optimal one-to-one assignment.

In [ ]:
# Fresh manager, run 2 frames so we have confirmed tracks, then inspect frame 2 cost matrix
rng_cost = default_rng(42)
per_frame_meas2 = mot_sim.generate_multi_target_measurements(trajectories, rng_cost)

mgr2 = TrackerManager(dt=1.0)
for k in range(2):
    mgr2.predict_all()
    zs = [np.array([r.range_m, r.azimuth_rad]) for r in per_frame_meas2[k]]
    mgr2.update(zs)

# Build frame-2 cost matrix without committing the update
mgr2.predict_all()
zs2 = [np.array([r.range_m, r.azimuth_rad]) for r in per_frame_meas2[2]]
cost = mgr2._build_cost_matrix(zs2)

print(f"Tracks: {[t.track_id for t in mgr2.tracks]}")
print(f"Measurements at frame 2: {len(zs2)} (includes true targets + possible clutter)")
print(f"\nCost matrix ({cost.shape[0]} tracks × {cost.shape[1]} measurements):")
print("  Values < 9.21 are within the chi-square gate; 1e9 = gated out")
with np.printoptions(precision=2, linewidth=120):
    display_cost = np.where(cost > 1e8, np.inf, cost)
    print(display_cost)

## The crossing at frame 49 — are two targets inside each other's gate?

This is the hardest frame for GNN. We run the full demo to frame 49 and then inspect which track-to-GT pairs are within the Mahalanobis gate.

In [ ]:
res = run_mot_demo(rng=default_rng(42), n_frames=100)

# Track positions at frame 49
frame49 = res["track_history"][49]
print("Confirmed track positions at frame 49:")
for tid, pos in sorted(frame49.items()):
    print(f"  Track {tid}: {pos.round(2)}")

print("\nGround-truth positions at frame 49:")
for i, traj in enumerate(trajectories):
    print(f"  GT {i+1}: {traj[49].round(2)}")

# Euclidean distance between every (track, GT) pair
print("\nEuclidean distance (m) — track vs GT at frame 49:")
tids = sorted(frame49.keys())
header = "         " + "  ".join(f"GT{i+1:2d}" for i in range(3))
print(header)
for tid in tids:
    row = f"Track {tid:2d}  "
    for i in range(3):
        d = np.linalg.norm(frame49[tid] - trajectories[i][49])
        row += f"{d:6.1f}  "
    print(row)

## Same track IDs before and after the crossing

If GNN assigns correctly at frame 49, the track-to-GT mapping must be the same at frames 48 and 52. An ID switch would show as a track changing its closest GT target between those frames.

In [ ]:
from scipy.optimize import linear_sum_assignment

def nearest_gt(frame_state, trajectories, k):
    """Return dict {track_id: gt_index} for the nearest GT per track at frame k."""
    if not frame_state:
        return {}
    tids = sorted(frame_state.keys())
    track_pos = np.array([frame_state[t] for t in tids])
    gt_pos = np.array([trajectories[j][k] for j in range(len(trajectories))])
    diff = track_pos[:, None, :] - gt_pos[None, :, :]
    cost = np.sqrt((diff ** 2).sum(axis=2))
    row_ind, col_ind = linear_sum_assignment(cost)
    return {tids[r]: col_ind[c] for r, c in zip(row_ind, range(len(col_ind))) if cost[r, c] < 50.0}

for k in (47, 48, 49, 50, 51, 52):
    mapping = nearest_gt(res["track_history"][k], trajectories, k)
    print(f"Frame {k}: {mapping}")

## Metrics — ID switches and per-track RMSE

In [ ]:
metrics = res["metrics"]
print(f"ID switches:   {metrics['id_switches']}")
print(f"Mean RMSE:     {metrics['mean_rmse']:.2f} m")
print()
# Track lifetimes
lifetime = {}
for frame in res["track_history"]:
    for tid in frame:
        lifetime[tid] = lifetime.get(tid, 0) + 1

print(f"{'Track':>7}  {'Lifetime':>9}  {'RMSE (m)':>10}")
print("-" * 32)
for tid in sorted(lifetime.keys()):
    rmse = metrics["per_track_rmse"].get(tid, float("nan"))
    label = "  ← GT-matched" if rmse < 5.0 and lifetime[tid] >= 50 else ""
    print(f"{tid:>7}  {lifetime[tid]:>9}  {rmse:>10.2f}{label}")

## Inline trajectory plot

Grey dashed = ground truth; coloured = estimated tracks with uncertainty ellipses every 10 frames.

In [ ]:
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

fig = plot_mot_results(res, show=False)
plt.tight_layout()
plt.show()